In [3]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Load dataset
df = pd.read_csv('/content/Chronic_Kidney_Dsease_data.csv')

# Replace missing values
df = df.fillna(df.mean(numeric_only=True))

# Check for remaining missing values
missing = df.isnull().sum()
print("\nMissing values after mean replacement:")
print(missing)


Missing values after mean replacement:
PatientID                        0
Age                              0
Gender                           0
Ethnicity                        0
SocioeconomicStatus              0
EducationLevel                   0
BMI                              0
Smoking                          0
AlcoholConsumption               0
PhysicalActivity                 0
DietQuality                      0
SleepQuality                     0
FamilyHistoryKidneyDisease       0
FamilyHistoryHypertension        0
FamilyHistoryDiabetes            0
PreviousAcuteKidneyInjury        0
UrinaryTractInfections           0
SystolicBP                       0
DiastolicBP                      0
FastingBloodSugar                0
HbA1c                            0
SerumCreatinine                  0
BUNLevels                        0
GFR                              0
ProteinInUrine                   0
ACR                              0
SerumElectrolytesSodium          0
SerumElectrolyt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install boruta

In [ ]:
!pip install catboost

In [ ]:
# Install required packages (run once)
!pip install catboost shap --quiet

# Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
import shap

# Load dataset
df = pd.read_csv('/content/Chronic_Kidney_Dsease_data.csv')

# Separate features and target
X = df.drop('Diagnosis', axis=1)
y = df['Diagnosis']

# Handle categorical columns by one-hot encoding
# Identify non-numeric columns in X
categorical_cols = X.select_dtypes(include=['object']).columns

# Apply one-hot encoding
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True) # drop_first to avoid multicollinearity

# Step 1: Remove highly correlated features (threshold > 0.9)
corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]
X_reduced = X.drop(columns=to_drop)

print(f" Dropped {len(to_drop)} highly correlated features: {to_drop}")

# Step 2: Train CatBoost model
model = CatBoostClassifier(verbose=0, random_state=42)
model.fit(X_reduced, y)

# Step 3: SHAP feature importance
explainer = shap.Explainer(model)
shap_values = explainer(X_reduced)

# Step 4: Rank features by mean absolute SHAP values
shap_importance = np.abs(shap_values.values).mean(axis=0)
shap_feature_importance = pd.Series(shap_importance, index=X_reduced.columns)
top_features = shap_feature_importance.sort_values(ascending=False).head(15).index.tolist()

print("\n Top 15 features by SHAP:")
print(top_features)

# Step 5: Create final dataset and save
X_final = X_reduced[top_features]
df_final = X_final.copy()
df_final['Diagnosis'] = y.values

df_final.to_csv('/content/CKD_SHAP_Top_Features.csv', index=False)
print("\n Saved: CKD_SHAP_Top_Features.csv")

In [ ]:
print(y.value_counts())

In [ ]:
# Install required libraries (if not already)
!pip install imbalanced-learn catboost --quiet

# Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import AdaBoostClassifier, ExtraTreesClassifier, RandomForestClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('/content/CKD_SHAP_Top_Features.csv')
X = df.drop('Diagnosis', axis=1)
y = df['Diagnosis']

# Split original (imbalanced) dataset
X_train_orig, X_test_orig, y_train_orig, y_test_orig = train_test_split(X, y, test_size=0.2, random_state=42)

# Apply SMOTE only for selected models
smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X, y)
X_train_sm, X_test_sm, y_train_sm, y_test_sm = train_test_split(X_smote, y_smote, test_size=0.2, random_state=42)

# Scale both sets
scaler_orig = StandardScaler()
X_train_orig_scaled = scaler_orig.fit_transform(X_train_orig)
X_test_orig_scaled = scaler_orig.transform(X_test_orig)

scaler_smote = StandardScaler()
X_train_sm_scaled = scaler_smote.fit_transform(X_train_sm)
X_test_sm_scaled = scaler_smote.transform(X_test_sm)

# Define models with SMOTE (trained on balanced data)
models_with_smote = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=20, min_samples_split=5,
        max_features='sqrt', bootstrap=True, random_state=42
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300, max_depth=20, min_samples_split=5,
        random_state=42
    ),
    "CatBoost": CatBoostClassifier(
        iterations=500, learning_rate=0.05, depth=6, l2_leaf_reg=3,
        loss_function='Logloss', verbose=0, random_state=42
    )
}

# Define models without SMOTE (trained on original data)
models_without_smote = {
    "Naive Bayes": GaussianNB(),
    "Logistic Regression": LogisticRegression(),
    "KNN": KNeighborsClassifier(),
    "AdaBoost": AdaBoostClassifier(
        n_estimators=200, learning_rate=0.5, random_state=42
    )
}

# Store results
results = []

# Train and evaluate models
for name, model in models_with_smote.items():
    if name == "CatBoost":
        model.fit(X_train_sm, y_train_sm)
        y_pred = model.predict(X_test_sm)
    else:
        model.fit(X_train_sm_scaled, y_train_sm)
        y_pred = model.predict(X_test_sm_scaled)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_sm, y_pred),
        "Precision": precision_score(y_test_sm, y_pred),
        "Recall": recall_score(y_test_sm, y_pred),
        "F1-Score": f1_score(y_test_sm, y_pred)
    })

# Train and evaluate models
for name, model in models_without_smote.items():
    if name == "Naive Bayes":
        model.fit(X_train_orig, y_train_orig)
        y_pred = model.predict(X_test_orig)
    else:
        model.fit(X_train_orig_scaled, y_train_orig)
        y_pred = model.predict(X_test_orig_scaled)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_orig, y_pred),
        "Precision": precision_score(y_test_orig, y_pred),
        "Recall": recall_score(y_test_orig, y_pred),
        "F1-Score": f1_score(y_test_orig, y_pred)
    })

# Display results
results_df = pd.DataFrame(results)
print("\n Final Model Performance:")
print(results_df.sort_values(by="Accuracy", ascending=False).reset_index(drop=True))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.0 MB/s eta 0:00:00


FileNotFoundError: [Errno 2] No such file or directory: '/content/CKD_SHAP_Top_Features.csv'

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# Predict probabilities for the positive class
y_prob = models_without_smote["Logistic Regression"].predict_proba(X_test_orig_scaled)[:, 1]

# Make sure the positive label is correctly set (check your dataset's positive class name)
positive_label = y_test_orig.unique()[1]  # or explicitly set, e.g., 'ckd'

# Compute ROC curve
fpr, tpr, thresholds = roc_curve(y_test_orig, y_prob, pos_label=positive_label)

# Compute AUC directly
roc_auc = roc_auc_score(y_test_orig, y_prob)

# Plot ROC curve
plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

print(f"AUC Score: {roc_auc:.4f}")

In [ ]:
from sklearn.metrics import roc_auc_score

# Get the Logistic Regression model from your dictionary
log_reg_model = models_without_smote["Logistic Regression"]

# Predict probabilities for the positive class
y_prob = log_reg_model.predict_proba(X_test_orig_scaled)[:, 1]

# Calculate AUC
auc_score = roc_auc_score(y_test_orig, y_prob)
print(f"AUC Score: {auc_score:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Example dictionary of models and their accuracies
# Replace with your actual accuracy values
model_accuracies = {
    'Logistic Regression': 0.92,
    'Extra Trees': 0.9885,
    'CatBoost': 0.9639,
    'Random Forest': 0.95,
    'XGBoost': 0.94
}

# Extract names and scores
models = list(model_accuracies.keys())
accuracies = list(model_accuracies.values())

# Plot
plt.figure(figsize=(8, 5))
bars = plt.bar(models, accuracies, color='skyblue', edgecolor='black')

# Add accuracy labels on top of bars
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
             f"{acc:.2%}", ha='center', fontsize=10)

plt.ylim(0, 1.05)
plt.ylabel('Accuracy')
plt.xlabel('Models')
plt.title('Model Accuracy Comparison')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()
